# Tool 5: Sleep macrostructure, the classical PSG summary metrics

Computes, for every recording of a database, the metrics that summarise the **architecture of the
night**, the ones printed on a clinical PSG report:

- **time in bed, sleep onset latency, sleep period time, total sleep time, wake after sleep onset,
  sleep efficiency**,
- the **latency to each stage** (N1, N2, N3, REM),
- the **duration and percentage of each stage**,
- the **event indices** per hour of sleep: apnea–hypopnea index (with its obstructive / central / mixed /
  hypopnea sub-indices), periodic limb movement index, arousal index,
- optionally the **oxygen desaturation index** (from the scored desaturations), **T90** (time with
  SpO2 below 90 % in bed) and **T88** (time with SpO2 <= 88 % during sleep, the ICSD-3-TR sleep-related
  hypoxemia criterion), both from the SpO2 signal.

**Inputs**: the remapped 30-s hypnogram written by tool 3 (`{file_id}_Hypnogram_remapped.txt`, one
AASM label per line), the scored events harmonised by tool 4 (`config_param/event_remap.json`), and the
EDF **header** (start time, duration, channel names). The EEG signal is **never** loaded. The only signal
reads are the optional `Light` channel (lights-off/on detection) and the `SpO2` channel (T90 / T88).

This tool is the one exception to the *feature tools start from the clean epochs* rule: macrostructure
needs the **complete** scored hypnogram, not the epochs that survived artefact rejection.

**Lights-off / lights-on** may come from various sources. The tool resolves them, in
order, from a `{file_id}_Summary_Export.txt` Profusion export (`Luminosité` column), the EDF `Light`
channel, two columns of the participant table, and finally the start/end of the recording, with a
warning whenever the fallback is used. Every result carries its `lights_source`.

**Outputs**: one long-format `{file_id}_sleep_metrics.tsv` (+ a `_checks.tsv` with the consistency
checks) per recording under `derivatives/features_macrostructure/`, and the database-level
`global_sleep_metrics.tsv` (one row per participant), an Excel workbook and an HTML report under
`reports_features_macrostructure/`. A last section compares the results with an external reference table
(e.g. the values of the clinical reports).

Work through the sections in order: **1** pick the study folder and scan, **2** set the parameters,
**3** choose the participants, **4** run, **5** (optional) compare with a reference table.
The **metric glossary** just below defines every metric and gives indicative normal ranges.

In [ ]:
# =============================================================================
# Tool 5 - Sleep macrostructure: setup, constants and shared configuration
# =============================================================================
import os
import re
import io
import json
import difflib
import datetime
import unicodedata
import warnings
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import mne
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from ipyfilechooser import FileChooser

# Shared participant handling (tools/participant_selection_lib.py: exclusion registry + participant
# selector), probed from the repo root (the canonical launch folder) or from tools/, so the import works
# whatever folder Voila is launched from.
import sys
for _p in [os.getcwd(), os.path.join(os.getcwd(), 'tools'),
           os.path.join(os.path.dirname(os.getcwd()), 'tools'), os.path.join(os.getcwd(), '..', 'tools')]:
    if os.path.isfile(os.path.join(_p, 'participant_selection_lib.py')):
        if _p not in sys.path:
            sys.path.insert(0, _p)
        break
import participant_selection_lib as PSL

warnings.filterwarnings('ignore')
mne.set_log_level('ERROR')
matplotlib.rcParams['figure.max_open_warning'] = 0

# Output folder names (data vs reports split, toolkit convention)
DATA_DIRNAME = 'features_macrostructure'
REPORTS_DIRNAME = 'reports_features_macrostructure'

# The constants, computations and figures of this tool live in tools/macrostructure_lib.py, shared with
# tools 5bis and 0 (tool-0 redesign): moved there verbatim, this tool calls the very same functions. The
# scored-event chain is edf_io_lib's (the same file as the library, identical code).
from macrostructure_lib import (
    checks_df, checks_summary_html, classify_stages, close_figs, compare_with_reference,
    compute_event_indices, compute_macrostructure, compute_odi, compute_recording, compute_t90,
    COST_EVENTS, COST_HEADER, COST_HYPNO, COST_LIGHTS, COST_SPO2, COST_WRITE, DEFAULT_EPOCH_S,
    default_tolerance, event_companion_paths, EVENT_COUNT_WINDOW, find_channel, glossary_html,
    guess_metric_mapping, guidance_html, HAS_OPENPYXL, LIGHT_CH_PATTERN, load_custom_stages,
    load_event_remap, load_events, load_hypnogram, load_spo2_signal, load_subject_info, long_to_wide,
    METRIC_INFO, METRIC_KEYS, METRICS, metrics_long_df, norm, ONSET_RULES, OTHER_STAGE_POLICY,
    params_html, parse_custom_field, participants_html, plot_comparison, plot_lights_overview,
    plot_metric_distributions, read_edf_header_info, reconcile_hypno_length, REFERENCE_RANGES,
    references_html, resolve_lights, run_checks, save_report_html, select_events_in_window,
    SPO2_CH_PATTERN, STATUS_COLORS)


## Metric glossary

All metrics are computed on the epochs between **lights-off** and **lights-on** (epoch length 30 s by
default, `ep` = epoch length in minutes). Epoch indices are 0-based. Lights-on is the first epoch *out*
of bed. "Sleep" = N1, N2, N3, R. The indicative ranges are values for **healthy adults**: they
shift with age, medication and the first-night effect, and are only meant as a quick sanity check
(they can be edited in the `REFERENCE_RANGES` dictionary above). Where each value comes from: the
collapsible *References* section at the end of the notebook.

| Metric | Definition / formula | Typical use | Indicative range (healthy adults) |
|---|---|---|---|
| **TIB**: time in bed | lights-off → lights-on: `n epochs in bed × ep` | Denominator of sleep efficiency | — |
| **Sleep onset** | first epoch after lights-off scored N1/N2/N3/R (default rule). Alternatives: first N2 epoch, first of 3 consecutive sleep epochs | Anchor for SOL, SPT and every latency | — |
| **SOL**: sleep onset latency | lights-off → sleep onset: `onset index × ep` | Sleep initiation (insomnia, hypersomnia: short SOL on an MSLT) | ~15–20 min at every adult age, > 30 min prolonged |
| **SPT**: sleep period time | sleep onset → last sleep epoch (both included) | Window of the sleep episode | — |
| **TST**: total sleep time | `(n N1 + n N2 + n N3 + n R within SPT) × ep` | Denominator of every per-hour index and stage % | 6–8 h (lab: ~7.5 h at 20–25 y → ~6 h after 75 y) |
| **WASO**: wake after sleep onset | `n W epochs within SPT × ep` | Sleep maintenance | < 30 min in young adults, +10 min per decade from 30 y |
| **Other within SPT** | MT / custom / unknown epochs within SPT | Reported so that `SPT = TST + WASO + other` | 0 |
| **SE**: sleep efficiency | `100 × TST / TIB` | Global sleep quality, the insomnia criterion | > 85 % (~80 % after 75 y) |
| **SME**: sleep maintenance efficiency | `100 × TST / SPT` | Continuity once asleep (independent of SOL) | > 90 % (convention) |
| **Latency to N1 / N2 / N3 / REM** | sleep onset → first epoch of the stage, undefined (NaN) if the stage never occurs | REM latency: narcolepsy (< 15 min = SOREMP) in night + MSLT, <50 min could be associated with depression | REM latency 50–110 min (~90 min at 25 y → ~55 min at 90 y) |
| **Stage duration** | `n epochs of the stage within SPT × ep` | Absolute amount of each stage | — |
| **Stage % of TST** | `100 × stage duration / TST` | Sleep architecture, sums to 100 % | N1 2–8 %, N2 45–58 %, N3 13–23 % (young adults, 5–10 % after 65 y), REM 18–25 % |
| **AHI** and **OAI / CAI / MAI / HI** | `n events / TST (h)`, AHI = OAI + CAI + MAI + HI | Sleep apnea diagnosis and severity | AHI < 5 normal, 5–15 mild, 15–30 moderate, > 30 severe  |
| **PLM index** | `n periodic limb movements / TST (h)` | Periodic limb movement disorder | < 15/h (adults) |
| **Arousal index** | `n arousals / TST (h)` | Sleep fragmentation | < 15/h (convention) |
| **ODI**: oxygen desaturation index | `n desaturations (≥ 3 % or 4 %) / TST (h)` | Hypoxic burden of respiratory events, state the threshold | < 5/h (convention) |
| **T90** | time with SpO2 < 90 % in bed, in min and `% of TST` | Nocturnal hypoxaemia | < 10 % of TST (research convention, not an ICSD criterion) |
| **T88**: during sleep | time with SpO2 ≤ 88 % on the sleep epochs (N1/N2/N3/R) in bed, in min | Sleep-related hypoxemia (ICSD-3-TR, criterion A: ≥ 5 min) | < 5 min |

**Consistency checks** run on every recording (written to `{file_id}_sleep_metrics_checks.tsv` and
summarised in the report): `SPT = TST + WASO + other`, `0 ≤ SE ≤ 100`, `TST ≤ SPT ≤ TIB`,
`SOL + SPT ≤ TIB`, stage % sum to 100, latencies within `[0, SPT]`, hypnogram length vs EDF duration,
lights file length vs hypnogram, lights-off < lights-on, event onsets inside the recording, unmapped
event labels, MT / custom stages present, no sleep at all.

**MT and custom stages** (YASA convention): counted in TIB and in SPT, excluded from TST and WASO. The
tool warns when it meets them. If you prefer them scored as a neighbouring stage, remap them with tool 3
(or set `OTHER_STAGE_POLICY = 'wake'` above to count them as wake).

**Lights-off / lights-on detection** (`summary txt` and `Light channel` sources): lights-off = the first
epoch after the **last lights-ON epoch that precedes the first sleep epoch**. Lights-on = the **first
lights-ON epoch after the last sleep epoch**. Without an ON epoch on either side the recording start /
end is used instead (warning). ON epochs inside the sleep period are counted and reported, not used.

In [ ]:
# The computations of this tool (header, hypnogram, lights, macrostructure, events, ODI, T90, checks) now
# live in tools/macrostructure_lib.py, imported in the first cell.


In [ ]:
# The figures and HTML helpers of this tool now live in tools/macrostructure_lib.py, imported in the first
# cell.


## 1. Folders and scan

Point the tool at the **study folder** holding the EDF files (scanned recursively) with, next to each EDF,
the remapped hypnogram from tool 3 and (when available) the scored-event companions
(`*_ScoredEvents_Export.txt`, `*_event_xml.csv`, `*.edf.XML`) and the Profusion
`*_Summary_Export.txt` (lights-off/on). The suffixes are auto-detected and editable.

`event_remap.json` (tool 4) is picked up from `<data>/config_param/` when present. Without it the
event indices (AHI, PLM, arousals, ODI) are NaN. The **output folder** is optional (default = the data
folder): the tool writes `derivatives/features_macrostructure/` and `reports_features_macrostructure/`
under it. The **participant table** is optional: its rows are joined to the global table on the column
you choose, and two of its columns can supply lights-off / lights-on (see Section 2).

**Scan** lists the participants, what was found next to each EDF, and how many are already processed.

In [ ]:
# =============================================================================
# Section 1 - folders, suffix auto-detection, scan
# =============================================================================
S = {}          # shared state across sections
_LBL = {'description_width': 'initial'}

fc_data = FileChooser(os.getcwd())
fc_data.show_only_dirs = True
fc_data.title = '<b>Study folder</b> (EDF files + hypnograms + event companions, scanned recursively):'

fc_out = FileChooser(os.getcwd())
fc_out.show_only_dirs = True
fc_out.title = ('<b>Output folder</b>: optional, default = the study folder (writes '
                '<code>derivatives/features_macrostructure/</code> and '
                '<code>reports_features_macrostructure/</code> under it):')
lbl_out_warn = widgets.HTML('')

fc_remap = FileChooser(os.getcwd())
fc_remap.filter_pattern = ['*.json']
fc_remap.title = ('<b>Event remap</b> (<code>config_param/event_remap.json</code> from tool 4) &mdash; '
                  'auto-selected when present under the study folder:')

fc_subj = FileChooser(os.getcwd())
fc_subj.filter_pattern = ['*.csv', '*.tsv', '*.xlsx']
fc_subj.title = ('<b>Participant info</b> (optional <code>.csv</code>/<code>.tsv</code>/<code>.xlsx</code> '
                 'joined to the global table; may hold the lights-off/on columns):')
dd_join_col = widgets.Dropdown(description='Join on:', options=[], value=None,
                               layout=widgets.Layout(width='320px', display='none'), style=_LBL)

txt_hypno_suffix = widgets.Text(description='Hypnogram suffix:', value='_Hypnogram_remapped.txt',
                                layout=widgets.Layout(width='420px'), style=_LBL)
lbl_hypno_suffix = widgets.HTML('')
txt_summary_suffix = widgets.Text(description='Lights (summary) txt suffix:', value='_Summary_Export.txt',
                                  layout=widgets.Layout(width='420px'), style=_LBL)
lbl_summary_suffix = widgets.HTML('')
txt_evt_txt_suffix = widgets.Text(description='Event TXT suffix:', value='_ScoredEvents_Export.txt',
                                  layout=widgets.Layout(width='420px'), style=_LBL)
lbl_evt_txt_suffix = widgets.HTML('')
txt_evt_csv_suffix = widgets.Text(description='Event CSV suffix:', value='_event_xml.csv',
                                  layout=widgets.Layout(width='420px'), style=_LBL)
lbl_evt_csv_suffix = widgets.HTML('')
txt_custom = widgets.Text(description='Custom stages:', value='',
                          placeholder='comma-separated non-AASM labels kept by tool 3, e.g. N4',
                          layout=widgets.Layout(width='520px'), style=_LBL)

btn_scan = widgets.Button(description='Scan', button_style='primary', icon='search')
lbl_scan = widgets.HTML('<i>Pick the study folder, then Scan.</i>')
out_scan = widgets.Output()


def _list_edf(folder):
    return [f for f in sorted(Path(folder).rglob('*'))
            if f.suffix.lower() == '.edf' and not f.name.startswith('._')]


def _suffix_counts(edf_files, files):
    """{suffix: number of EDFs having a companion <stem><suffix>} over `files`."""
    counts = {}
    for edf in edf_files:
        for f in files:
            if norm(f.name).startswith(norm(edf.stem)):
                suf = f.name[len(edf.stem):]
                counts[suf] = counts.get(suf, 0) + 1
    return counts


def _detect_suffixes(edf_folder):
    """Auto-fill the four suffix fields from the files found next to the EDFs (tool 7's rules:
    hypnogram = longest suffix among those present for >= 50 % of the max count, skipping the
    event / summary / study-log exports; the exports = most frequent suffix, shortest on ties)."""
    edf_files = _list_edf(edf_folder)
    n_total = len(edf_files)
    if not edf_files:
        lbl_hypno_suffix.value = '<small style="color:#888">No EDF file found (recursive scan).</small>'
        return
    all_txt = [f for f in Path(edf_folder).rglob('*') if f.suffix.lower() == '.txt']
    all_csv = [f for f in Path(edf_folder).rglob('*') if f.suffix.lower() == '.csv']

    def fill(widget, label, counts, what, pick):
        if not counts:
            label.value = f'<small style="color:#888">No {what} detected next to the EDF files.</small>'
            return
        best, n = pick(counts)
        widget.value = best
        parts = [f'<b>{s}</b>&nbsp;(&times;{c}){"&nbsp;&larr; selected" if s == best else ""}'
                 for s, c in sorted(counts.items(), key=lambda x: -x[1])]
        color = '#2e7d32' if n == n_total else '#e67e00'
        label.value = (f'<small style="color:{color}">{what} detected: {"&nbsp;&middot;&nbsp;".join(parts)}'
                       f': {n}/{n_total} files matching</small>')

    most_frequent = lambda c: min(c.items(), key=lambda x: (-x[1], len(x[0])))
    # hypnogram: exclude the exports from the SELECTION only, every suffix stays listed
    hyp_counts = _suffix_counts(edf_files, all_txt)
    sel = {s: c for s, c in hyp_counts.items()
           if not any(tok in s.lower() for tok in ('event', 'summary', 'studylog'))} or hyp_counts

    def pick_hypno(_):
        mx = max(sel.values())
        cands = {s: c for s, c in sel.items() if c >= mx * 0.5}
        return max(cands.items(), key=lambda x: (len(x[0]), x[1]))
    fill(txt_hypno_suffix, lbl_hypno_suffix, hyp_counts, 'Hypnogram .txt', pick_hypno)
    fill(txt_summary_suffix, lbl_summary_suffix,
         _suffix_counts(edf_files, [f for f in all_txt if 'summary' in f.name.lower()]),
         'Lights (summary) txt', most_frequent)
    fill(txt_evt_txt_suffix, lbl_evt_txt_suffix,
         _suffix_counts(edf_files, [f for f in all_txt if 'event' in f.name.lower()]),
         'Event TXT export', most_frequent)
    fill(txt_evt_csv_suffix, lbl_evt_csv_suffix, _suffix_counts(edf_files, all_csv),
         'Event CSV', most_frequent)


def _on_data(chooser):
    """Point the other choosers into the study folder, auto-select event_remap.json, detect suffixes."""
    try:
        data = fc_data.selected_path
        if not data:
            return
        data = Path(data)
        fc_out.reset(path=str(data))
        _on_out()
        fc_subj.reset(path=str(data))
        remap_default = data / 'config_param' / 'event_remap.json'
        # the chooser is only pre-pointed; load_remap_file() falls back to this default
        # when the user does not click Select
        if remap_default.exists():
            fc_remap.reset(path=str(remap_default.parent), filename=remap_default.name)
        else:
            fc_remap.reset(path=str(data))
        cs = load_custom_stages(data)
        txt_custom.value = ', '.join(cs)
        _detect_suffixes(data)
        lbl_scan.value = '<i>Study folder set: check the suffixes, then Scan.</i>'
    except Exception as exc:
        lbl_scan.value = f'<span style="color:#c62828">Study folder error: {exc}</span>'


def _on_out(chooser=None):
    """Warn when the output folder is not the study folder: tool 5bis would then keep its exclusion
    registry beside tool 5's outputs, away from the study's config_param/ read by the other tools."""
    try:
        out, data = fc_out.selected_path, fc_data.selected_path
        if out and data and (os.path.normcase(str(Path(out).resolve()))
                             != os.path.normcase(str(Path(data).resolve()))):
            lbl_out_warn.value = ('<span style="color:#e65100">&#9888; The output folder is not the study '
                                  'folder: tool 5bis will keep its exclusion registry beside these outputs, '
                                  'away from the study\'s <code>config_param/</code> read by the other tools. '
                                  'Leave it empty unless you need a separate run.</span>')
        else:
            lbl_out_warn.value = ''
    except Exception as exc:
        lbl_out_warn.value = f'<span style="color:#c62828">Output folder check failed: {exc}</span>'


def _on_subj(chooser):
    """Load the optional participant table and offer its columns as the join key."""
    try:
        df, err = load_subject_info(fc_subj.selected)
        if df is None:
            S.pop('subj_info', None)
            dd_join_col.layout.display = 'none'
            lbl_scan.value = f'<span style="color:#ef6c00">Participant info unreadable: {err}</span>'
            return
        S['subj_info'] = df
        dd_join_col.options = list(df.columns)
        guess = [c for c in df.columns if norm(c) in ('file_id', 'fileid', 'sub_id', 'subid', 'id',
                                                       'participant', 'participant_id', 'subject')]
        dd_join_col.value = guess[0] if guess else list(df.columns)[0]
        dd_join_col.layout.display = ''
        lbl_scan.value = (f'<i>Participant info loaded ({len(df)} rows, columns: '
                          f'{", ".join(map(str, df.columns))}): check the join column, then Scan.</i>')
    except Exception as exc:
        lbl_scan.value = f'<span style="color:#c62828">Participant info error: {exc}</span>'


def subject_row(file_id):
    """The participant-table row of a file_id (dict) or None."""
    info = S.get('subj_info')
    col = dd_join_col.value
    if info is None or not col or col not in info.columns:
        return None
    hit = info[info[col].astype(str).str.strip().map(norm) == norm(file_id)]
    return hit.iloc[0].to_dict() if len(hit) else None


def load_remap_file():
    """event_remap.json from the chooser, else the default under the study folder. (dict|None, note)."""
    path = fc_remap.selected
    if not path and fc_data.selected_path:
        cand = Path(fc_data.selected_path) / 'config_param' / 'event_remap.json'
        path = str(cand) if cand.exists() else None
    if not path or not Path(path).exists():
        return None, 'no event_remap.json - event labels cannot be counted (run tool 4)'
    try:
        return load_event_remap(path), f'event remap: {path} ({len(load_event_remap(path))} labels)'
    except Exception as exc:
        return None, f'event_remap.json unreadable ({exc})'


def scan_folder(_=None):
    """List the EDFs, read each header (no signal), locate the companions, count the epochs of
    each hypnogram, and flag what is already processed. Everything here is non-fatal."""
    with out_scan:
        clear_output()
        try:
            if not fc_data.selected_path:
                lbl_scan.value = '<span style="color:#c62828">Select the study folder first.</span>'
                return
            data_root = Path(fc_data.selected_path)
            out_root = Path(fc_out.selected_path) if fc_out.selected_path else data_root
            data_dir = out_root / 'derivatives' / DATA_DIRNAME
            reports_dir = out_root / REPORTS_DIRNAME
            edf_files = _list_edf(data_root)
            if not edf_files:
                lbl_scan.value = '<span style="color:#c62828">No EDF file found under that folder.</span>'
                return
            custom = parse_custom_field(txt_custom.value) or load_custom_stages(data_root)
            txt_custom.value = ', '.join(custom)
            remap, remap_note = load_remap_file()
            hypno_lookup = {norm(p.name): p for p in data_root.rglob('*') if p.suffix.lower() == '.txt'}
            hsuf, ssuf = txt_hypno_suffix.value, txt_summary_suffix.value
            tsuf, csuf = txt_evt_txt_suffix.value, txt_evt_csv_suffix.value

            parts, rows, warn = [], [], []
            n_done, n_light, n_spo2 = 0, 0, 0
            for edf in edf_files:
                fid = edf.stem
                p = {'file_id': fid, 'edf': edf, 'subtree': edf.parent.relative_to(data_root)}
                try:
                    header = read_edf_header_info(edf)
                except Exception as exc:
                    header = None
                    warn.append(f'{fid}: EDF header unreadable ({exc}), will fail at run time')
                p['header'] = header
                hyp = hypno_lookup.get(norm(f'{fid}{hsuf}'))
                if hyp is None and (edf.with_name(f'{fid}{hsuf}')).exists():
                    hyp = edf.with_name(f'{fid}{hsuf}')
                p['hypno'] = hyp
                n_hyp = None
                if hyp is not None:
                    try:
                        n_hyp = len(load_hypnogram(hyp))
                    except Exception as exc:
                        warn.append(f'{fid}: hypnogram unreadable ({exc})')
                else:
                    warn.append(f'{fid}: no hypnogram <code>{fid}{hsuf}</code>, will be skipped')
                p['n_hypno'] = n_hyp
                summ = hypno_lookup.get(norm(f'{fid}{ssuf}'))
                p['summary_txt'] = summ
                txt, csv, xml = event_companion_paths(edf, tsuf, csuf)
                p.update({'evt_txt': txt, 'evt_csv': csv, 'evt_xml': xml})
                chs = header['ch_names'] if header else []
                p['light_ch'] = find_channel(chs, LIGHT_CH_PATTERN)
                p['spo2_ch'] = find_channel(chs, SPO2_CH_PATTERN)
                n_light += p['light_ch'] is not None
                n_spo2 += p['spo2_ch'] is not None
                p['out_data'] = data_dir / p['subtree']
                p['out_reports'] = reports_dir / p['subtree']
                done = (p['out_data'] / f'{fid}_sleep_metrics.tsv').exists()
                half = (not done) and (p['out_data'] / f'{fid}_sleep_metrics_checks.tsv').exists()
                p['done'] = done
                n_done += done
                if half:
                    warn.append(f'{fid}: checks file without metrics file (interrupted run) &mdash; '
                                f'will be reprocessed')
                n_edf_ep = int(np.floor(header['duration_s'] / DEFAULT_EPOCH_S)) if header else None
                if header and n_hyp is not None and abs(n_hyp - n_edf_ep) > 1:
                    warn.append(f'{fid}: hypnogram {n_hyp} epochs vs EDF {n_edf_ep}, mismatch > 1 '
                                f'epoch, will fail (wrong hypnogram or epoch length?)')
                lights_avail = ('txt' if summ is not None else
                                f'channel {p["light_ch"]}' if p['light_ch'] else
                                'table' if subject_row(fid) is not None else '&ndash; (bounds)')
                p['lights_avail'] = lights_avail
                ev_src = 'txt' if txt else 'csv' if csv else 'xml' if xml else '&ndash;'
                p['evt_avail'] = ev_src
                rows.append({'file_id': fid, 'subtree': str(p['subtree']),
                             'start': header['start_dt'].strftime('%Y-%m-%d %H:%M:%S') if header else '?',
                             'duration_min': round(header['duration_s'] / 60, 1) if header else np.nan,
                             'epochs_edf': n_edf_ep, 'epochs_hypno': n_hyp if n_hyp is not None else 'NO',
                             'lights': lights_avail, 'events': ev_src,
                             'spo2': p['spo2_ch'] or '&ndash;', 'processed': 'yes' if done else ''})
                parts.append(p)

            S.update({'parts': parts, 'data_root': data_root, 'out_root': out_root, 'data_dir': data_dir,
                      'reports_dir': reports_dir, 'custom_stages': custom, 'event_remap': remap,
                      'remap_note': remap_note})
            longest = max((len(str(p['out_data'] / f'{p["file_id"]}_sleep_metrics_checks.tsv'))
                           for p in parts), default=0)
            if longest > 245:
                warn.append(f'the longest output path is {longest} characters: close to the '
                            f'260-character Windows limit; move the dataset to a shorter path if files '
                            f'fail to be written.')
            if remap is None:
                warn.append(remap_note)
            display(HTML(f'<b>{len(parts)} recordings</b> &nbsp;|&nbsp; <b>{n_done} / {len(parts)}</b> '
                         f'already processed &nbsp;|&nbsp; Light channel in {n_light}, SpO2 in {n_spo2} '
                         f'&nbsp;|&nbsp; <small>{remap_note}</small>'))
            display(HTML(pd.DataFrame(rows).to_html(index=False, border=0, escape=False)))
            for w in warn:
                display(HTML(f'<span style="color:#ef6c00">&#9888; {w}</span>'))
            # pre-fill the Section-2 channel fields with what the files carry
            lights_names = sorted({p['light_ch'] for p in parts if p['light_ch']})
            spo2_names = sorted({p['spo2_ch'] for p in parts if p['spo2_ch']})
            txt_light_ch.value = lights_names[0] if lights_names else ''
            txt_spo2_ch.value = spo2_names[0] if spo2_names else ''
            build_participant_list()
            lbl_scan.value = (f'<span style="color:#2e7d32">Scan done, {len(parts)} recordings, '
                              f'{n_done} already processed.</span>')
        except Exception as exc:
            lbl_scan.value = f'<span style="color:#c62828">Scan failed: {exc}</span>'
            display(HTML(f'<pre style="color:#c62828">{exc}</pre>'))


fc_data.register_callback(_on_data)
fc_out.register_callback(_on_out)
fc_subj.register_callback(_on_subj)
btn_scan.on_click(scan_folder)

display(widgets.VBox([fc_data, fc_out, lbl_out_warn, fc_remap, fc_subj, dd_join_col,
                      txt_hypno_suffix, lbl_hypno_suffix, txt_summary_suffix, lbl_summary_suffix,
                      txt_evt_txt_suffix, lbl_evt_txt_suffix, txt_evt_csv_suffix, lbl_evt_csv_suffix,
                      txt_custom, widgets.HBox([btn_scan, lbl_scan]), out_scan]))

## 2. Parameters

**Lights-off / lights-on** decide the time-in-bed window. `auto` tries, in order, the Profusion summary
txt (`Luminosité` column, 1 = ON / 0 = OFF), the EDF `Light` channel, the two columns of the participant
table (clock time `HH:MM` or epoch index), and finally the recording start / end (always with a
warning). Force one source to check the others against it. The **sleep-onset rule** anchors SOL, SPT and
every latency. The default follows the definition file (first non-W epoch).

**ODI** and **T90** are opt-in: ODI needs the desaturation depth (read from the `.edf.XML` when the
events come from the txt/csv export), T90 reads the SpO2 signal of every recording: both make the run
slower.

In [ ]:
# =============================================================================
# Section 2 - parameters (lights, sleep onset, events, ODI, T90)
# =============================================================================
def _w(px):
    """A FRESH width Layout (never share a Layout object between widgets)."""
    return widgets.Layout(width=f'{px}px')


def _ind(px=14):
    return widgets.Layout(margin=f'0 0 0 {px}px')


def _desc(text, indent=14):
    return widgets.HTML(f'<div style="line-height:1.2;margin-left:{indent}px">'
                        f'<small style="color:#555">{text}</small></div>')


def _head(text):
    return widgets.HTML(f'<b style="font-size:110%">{text}</b>')


def _reveal(checkbox, *boxes):
    """Show/hide widgets from `checkbox`; the initial display is derived from the CURRENT value
    (an observe handler only fires on a change, so a pre-ticked box would stay hidden)."""
    for box in boxes:
        box.layout.display = '' if checkbox.value else 'none'

    def _on(change, boxes=boxes):
        for box in boxes:
            box.layout.display = '' if change['new'] else 'none'

    checkbox.observe(_on, names='value')


# ---------------------------------------------------------------------------
# 2a. Epoch length and lights
# ---------------------------------------------------------------------------
txt_epoch_s = widgets.IntText(description='Epoch length (s):', value=DEFAULT_EPOCH_S, layout=_w(220), style=_LBL)

LIGHTS_OPTIONS = [('auto (summary txt -> Light channel -> participant table -> recording bounds)', 'auto'),
                  ('summary txt (Profusion export)', 'summary_txt'),
                  ('EDF Light channel', 'light_channel'),
                  ('participant table columns', 'participant_table'),
                  ('recording start / end', 'recording_bounds')]
dd_lights_source = widgets.Dropdown(description='Lights-off/on source:', options=LIGHTS_OPTIONS,
                                    value='auto', layout=_w(620), style=_LBL)
txt_lights_col = widgets.Text(description='Summary txt column:', value='Luminosité', layout=_w(300), style=_LBL)
txt_light_ch = widgets.Text(description='Light channel:', value='', placeholder='auto (Light / Lights)',
                            layout=_w(300), style=_LBL)
txt_lights_off_col = widgets.Text(description='Table column lights-off:', value='lights_off',
                                  layout=_w(300), style=_LBL)
txt_lights_on_col = widgets.Text(description='Table column lights-on:', value='lights_on',
                                 layout=_w(300), style=_LBL)

box_lights = widgets.VBox([
    _head('Time in bed'),
    _desc('The hypnogram is assumed to start at the recording start, one label per epoch.'),
    widgets.HBox([txt_epoch_s], layout=_ind(14)),
    _desc('Lights-off / lights-on: each source is tried in the order shown; the first usable one wins '
          'and is recorded as <code>lights_source</code>. Recording start / end is the last resort and '
          'is always flagged. Summary txt and Light channel: 1 = lights ON, 0 = OFF; lights-off = first '
          'epoch after the last ON epoch before the first sleep epoch, lights-on = first ON epoch after '
          'the last sleep epoch.'),
    widgets.HBox([dd_lights_source], layout=_ind(14)),
    widgets.HBox([txt_lights_col, txt_light_ch], layout=_ind(14)),
    _desc('Participant table: clock time <code>HH:MM</code> / <code>HH:MM:SS</code> (rolled to the next '
          'day when after midnight) or a bare epoch index; a blank cell falls back to the recording bound.'),
    widgets.HBox([txt_lights_off_col, txt_lights_on_col], layout=_ind(14)),
])

# ---------------------------------------------------------------------------
# 2b. Sleep onset
# ---------------------------------------------------------------------------
dd_onset_rule = widgets.Dropdown(description='Sleep onset =', options=ONSET_RULES, value=ONSET_RULES[0],
                                 layout=_w(420), style=_LBL)
box_onset = widgets.VBox([
    _head('Sleep onset'),
    _desc('Anchor of SOL, SPT, TST and every stage latency. <i>first non-W epoch</i> is the definition '
          'of the formulas file (AASM); <i>first N2 epoch</i> and <i>3 consecutive sleep epochs</i> are '
          'common clinical variants. Under a variant the sleep epochs before the onset are reported but '
          'not counted.'),
    widgets.HBox([dd_onset_rule], layout=_ind(14)),
])

# ---------------------------------------------------------------------------
# 2c. Events, ODI, T90
# ---------------------------------------------------------------------------
cb_events = widgets.Checkbox(value=True, indent=False, layout=_w(620),
                             description='Event indices from the scored events (AHI + OAI/CAI/MAI/HI, PLM index, arousal index)')
cb_odi = widgets.Checkbox(value=False, indent=False, layout=_w(620),
                          description='ODI - oxygen desaturation index (slower: re-reads the .edf.XML for the depths)')
txt_desat_thr = widgets.FloatText(description='Desaturation depth >= (%):', value=3.0, layout=_w(260), style=_LBL)
box_odi = widgets.HBox([txt_desat_thr], layout=_ind(28))
cb_t90 = widgets.Checkbox(value=False, indent=False, layout=_w(720),
                          description='T90 / T88 - time with SpO2 below threshold (slower: reads the SpO2 signal of every recording)')
txt_spo2_ch = widgets.Text(description='SpO2 channel:', value='', placeholder='auto (SpO2 / SaO2)',
                           layout=_w(260), style=_LBL)
txt_spo2_thr = widgets.FloatText(description='SpO2 threshold (%):', value=90.0, layout=_w(220), style=_LBL)
txt_spo2_floor = widgets.FloatText(description='Artefact floor (%):', value=50.0, layout=_w(220), style=_LBL)
box_t90 = widgets.HBox([txt_spo2_ch, txt_spo2_thr, txt_spo2_floor], layout=_ind(28))
_reveal(cb_odi, box_odi)
_reveal(cb_t90, box_t90)

box_events = widgets.VBox([
    _head('Scored events and oxygen'),
    _desc('Events are counted between lights-off and lights-on (<code>EVENT_COUNT_WINDOW</code>) and '
          'divided by TST in hours. Labels come from <code>event_remap.json</code> (tool 4): '
          '<code>apnea_obstructive</code> / <code>apnea_central</code> / <code>apnea_mixed</code>, '
          '<code>hypopnea*</code>, <code>plm</code>, <code>arousal*</code>, <code>spo2_desaturation</code>. '
          'A label absent from the remap is NOT counted and is reported.'),
    widgets.VBox([cb_events], layout=_ind(14)),
    widgets.VBox([cb_odi], layout=_ind(14)),
    _desc('Report which threshold was used (3 % = current AASM recommendation, 4 % in older studies); '
          'without any depth information every scored desaturation is counted.', indent=28),
    box_odi,
    widgets.VBox([cb_t90], layout=_ind(14)),
    _desc('Samples <= 0 or below the artefact floor (sensor off, motion) are discarded and their '
          'percentage reported; T90 is given in minutes and as % of TST. T88 = time with SpO2 <= 88 % '
          '(fixed ICSD-3-TR adult threshold) on the sleep epochs only; >= 5 min meets criterion A of '
          'sleep-related hypoxemia.', indent=28),
    box_t90,
])

lbl_params = widgets.HTML()


def get_params():
    """Read every Section-2 widget into one config dict; returns (cfg, errors)."""
    errors = []
    epoch_s = int(txt_epoch_s.value)
    if epoch_s <= 0:
        errors.append('the epoch length must be positive')
    if cb_odi.value and float(txt_desat_thr.value) < 0:
        errors.append('the desaturation threshold must be >= 0')
    if cb_t90.value and not (0 < float(txt_spo2_thr.value) <= 100):
        errors.append('the SpO2 threshold must be in (0, 100]')
    if cb_t90.value and not (0 <= float(txt_spo2_floor.value) < float(txt_spo2_thr.value)):
        errors.append('the artefact floor must be below the SpO2 threshold')
    cfg = dict(epoch_s=epoch_s, lights_source=dd_lights_source.value,
               lights_col=txt_lights_col.value.strip() or 'Luminosité',
               light_ch=txt_light_ch.value.strip(),
               lights_off_col=txt_lights_off_col.value.strip() or 'lights_off',
               lights_on_col=txt_lights_on_col.value.strip() or 'lights_on',
               onset_rule=dd_onset_rule.value, events=bool(cb_events.value),
               odi=bool(cb_odi.value), odi_thr=float(txt_desat_thr.value),
               t90=bool(cb_t90.value), spo2_ch=txt_spo2_ch.value.strip(),
               spo2_thr=float(txt_spo2_thr.value), spo2_floor=float(txt_spo2_floor.value),
               other_stage_policy=OTHER_STAGE_POLICY, event_count_window=EVENT_COUNT_WINDOW)
    return cfg, errors


display(widgets.VBox([box_lights, box_onset, box_events, lbl_params]))

## 3. Participants

Run **all participants** (the already-processed ones are skipped while *Skip already processed* is
ticked) or **a subset**: type an id in the add box (it autocompletes with the participants not selected
yet, Enter adds it), paste a list, or pick ids in the participant list (which also shows who is already processed). A recording counts as **already
processed** when its `{file_id}_sleep_metrics.tsv` is on disk. *Skip already processed* applies to a subset
too: an orange box names the selected recordings it will leave out, untick it to reprocess them. The
global tables are always rebuilt from every per-recording file present, so partial runs accumulate.

In [ ]:
# =============================================================================
# Section 3 - participant selection: all participants or a subset (shared selector)
# =============================================================================
participant_selector = PSL.ParticipantSelector()
cb_skip = participant_selector.cb_skip       # same widget, name kept for the run cell
lbl_part = widgets.HTML('')


def build_participant_list():
    """Hand the scanned recordings to the selector, with the already-processed ones. Tool 5 extracts the
    raw macrostructure of EVERY recording and knows nothing of exclusions: those are decided (and shown)
    by tool 5bis, from this tool's table."""
    parts = S.get('parts', [])
    participant_selector.set_participants([p['file_id'] for p in parts],
                                          done=[p['file_id'] for p in parts if p.get('done')])


def selected_participants():
    """The recordings to run: the selector's choice, minus the already-processed ones when skipping."""
    keep = {PSL.id_key(f) for f in participant_selector.selected_ids()}
    return [p for p in S.get('parts', []) if PSL.id_key(p['file_id']) in keep]


display(widgets.VBox([participant_selector.widget, lbl_part]))


## 4. Run

For each recording: read the EDF header, load and check the hypnogram, resolve lights-off/on, compute
the macrostructure, count the scored events (and optionally ODI / T90), run the consistency checks,
then write `{file_id}_sleep_metrics_checks.tsv` and `{file_id}_sleep_metrics.tsv`. A recording that
fails (no hypnogram, length mismatch) is listed in `sleep_metrics_failed.tsv` and never stops the run.
The database tables, the workbook and the report are then rebuilt from every per-recording file on disk.

In [ ]:
# =============================================================================
# Section 4 - run: per recording, then the database-level outputs
# =============================================================================
EXCEL_MAX_ROWS = 1048575

btn_run = widgets.Button(description='Run', button_style='success', icon='play')
lbl_run = widgets.HTML()
progress_part = widgets.IntProgress(value=0, min=0, max=1, description='Participants:',
                                    layout=widgets.Layout(width='620px'), style=_LBL)
progress_step = widgets.IntProgress(value=0, min=0, max=1, bar_style='info', description='Pipeline:',
                                    layout=widgets.Layout(width='620px'), style=_LBL)
lbl_phase = widgets.HTML()
out_run = widgets.Output()

_STEP_LEGEND = widgets.HTML(
    '<div style="width:620px;font-size:85%;color:#555;display:flex;text-align:center">'
    + ''.join(f'<div style="flex:{c};border-right:1px solid #ccc">{n}</div>'
              for n, c in [('Header', COST_HEADER), ('Hypnogram', COST_HYPNO), ('Lights', COST_LIGHTS),
                           ('Events', COST_EVENTS), ('SpO2', COST_SPO2), ('Write', COST_WRITE)])
    + '</div>')


def _write_tsv(df, path):
    """Write a table, creating the folder. Empty tables are skipped."""
    if df is None or len(df) == 0:
        return None
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, sep='\t', index=False)
    return path


def _join_subject_info(df):
    """Left-join the optional participant table on file_id (normcase-insensitive on both sides)."""
    info = S.get('subj_info')
    col = dd_join_col.value
    if df is None or len(df) == 0 or info is None or not col or col not in info.columns:
        return df
    try:
        left = df.copy()
        left['_key'] = left['file_id'].astype(str).map(norm)
        right = info.copy()
        right['_key'] = right[col].astype(str).str.strip().map(norm)
        right = right.drop_duplicates('_key')
        right = right.drop(columns=[c for c in right.columns if c in left.columns and c != '_key'])
        return left.merge(right, on='_key', how='left').drop(columns=['_key'])
    except Exception:
        return df


def process_participant(p, cfg, log):
    """Full pipeline for one recording. Returns a one-line summary; a fatal step raises."""
    fid = p['file_id']
    steps = (COST_HEADER + COST_HYPNO + COST_LIGHTS + COST_WRITE
             + (COST_EVENTS if (cfg['events'] or cfg['odi']) else 0) + (COST_SPO2 if cfg['t90'] else 0))
    progress_step.max = steps
    progress_step.value = 0

    def phase(name, add):
        lbl_phase.value = f'<small>{fid}: {name}</small>'
        progress_step.value = min(progress_step.value + add, steps)

    # The computation is shared with tool 0 (macrostructure_lib.compute_recording: the former lines of
    # this function, moved verbatim); this function keeps the progress bar, the writing and the log.
    r = compute_recording(p, cfg, S.get('custom_stages', []), S.get('event_remap'), subject_row(fid),
                          txt_hypno_suffix.value, txt_evt_txt_suffix.value, txt_evt_csv_suffix.value,
                          phase)
    metrics, checks, provenance, lights = r['metrics'], r['checks'], r['provenance'], r['lights']
    out_data = p['out_data']
    _write_tsv(checks_df(fid, checks), out_data / f'{fid}_sleep_metrics_checks.tsv')
    _write_tsv(metrics_long_df(fid, metrics, provenance), out_data / f'{fid}_sleep_metrics.tsv')
    progress_step.value = steps

    def fmt(k):
        v = metrics.get(k, np.nan)
        return f'{v:.1f}' if v is not None and np.isfinite(v) else 'NaN'
    summary = (f'TIB {fmt("tib_min")} &middot; SOL {fmt("sol_min")} &middot; TST {fmt("tst_min")} &middot; '
               f'WASO {fmt("waso_min")} &middot; SE {fmt("se_pct")} % &middot; REM lat {fmt("lat_rem_min")}'
               + (f' &middot; AHI {fmt("ahi")}' if cfg['events'] else '')
               + (f' &middot; ODI {fmt("odi")}' if cfg['odi'] else '')
               + (f' &middot; T90 {fmt("t90_min")} min' if cfg['t90'] else '')
               + f' &middot; lights: {lights["source"]}')
    for c in checks:
        if c['status'] in ('warning', 'fail'):
            log(f'  <span style="color:{STATUS_COLORS[c["status"]]}">&#9888; {c["check"]}: {c["detail"]}</span>')
    return summary


def rebuild_globals(cfg, log):
    """Rebuild the database tables by globbing the per-recording files from disk."""
    data_dir, reports_dir = S['data_dir'], S['reports_dir']
    reports_dir.mkdir(parents=True, exist_ok=True)
    frames, cframes = [], []
    for f in sorted(data_dir.rglob('*_sleep_metrics.tsv')):
        if norm(f.name).startswith(norm('global_')):
            continue
        try:
            # file_id forced to str: a numeric id ("73") would otherwise come back as an int
            frames.append(pd.read_csv(f, sep='\t', dtype={'file_id': str, 'value': str}, keep_default_na=False))
        except Exception as exc:
            log(f'&#9888; could not read {f.name}: {exc}')
    for f in sorted(data_dir.rglob('*_sleep_metrics_checks.tsv')):
        try:
            cframes.append(pd.read_csv(f, sep='\t', dtype={'file_id': str}))
        except Exception as exc:
            log(f'&#9888; could not read {f.name}: {exc}')
    long_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    wide = long_to_wide(long_df)
    wide = _join_subject_info(wide)
    checks = pd.concat(cframes, ignore_index=True) if cframes else pd.DataFrame(
        columns=['file_id', 'check', 'status', 'detail'])
    _write_tsv(wide, reports_dir / 'global_sleep_metrics.tsv')
    _write_tsv(checks, reports_dir / 'global_sleep_metrics_checks.tsv')
    return {'wide': wide, 'checks': checks}


def glossary_df():
    rows = []
    for key, label, unit, group, definition in METRICS:
        rng = REFERENCE_RANGES.get(key)
        rows.append({'metric': key, 'label': label, 'unit': unit, 'group': group, 'definition': definition,
                     'indicative_low': rng[0] if rng else np.nan, 'indicative_high': rng[1] if rng else np.nan,
                     'note': rng[2] if rng else ''})
    return pd.DataFrame(rows)


def write_excel(tables, cfg, failed, path, log):
    """One workbook, one sheet per table (the TSV files stay the source of truth)."""
    if not HAS_OPENPYXL:
        log('&#9888; openpyxl is missing - the .xlsx workbook was not written (the TSV tables are complete)')
        return None
    par = pd.DataFrame({'parameter': list(cfg.keys()), 'value': [str(v) for v in cfg.values()]})
    sheets = {'sleep_metrics': tables.get('wide'), 'checks': tables.get('checks'),
              'glossary': glossary_df(), 'parameters': par,
              'failed': pd.DataFrame(failed, columns=['file_id', 'reason']) if failed else pd.DataFrame()}
    try:
        with pd.ExcelWriter(path, engine='openpyxl') as xl:
            for name, df in sheets.items():
                if df is None or len(df) == 0:
                    pd.DataFrame({'note': ['(empty)']}).to_excel(xl, sheet_name=name, index=False)
                    continue
                if len(df) > EXCEL_MAX_ROWS:
                    log(f'&#9888; sheet "{name}" truncated to {EXCEL_MAX_ROWS} rows - use the TSV')
                    df = df.head(EXCEL_MAX_ROWS)
                df.to_excel(xl, sheet_name=name, index=False)
        return path
    except Exception as exc:
        log(f'&#9888; could not write the workbook: {exc}')
        return None


def build_database_report(tables, cfg, path, comparison=None):
    """Database-level report: summary, lights overview, metric distributions with the indicative
    ranges, checks, participant table, optional comparison, glossary, and "where to look"."""
    wide, checks = tables.get('wide'), tables.get('checks')
    n_part = len(wide) if wide is not None else 0
    src_counts = wide['lights_source'].value_counts().to_dict() if n_part else {}
    n_warn = int((pd.to_numeric(wide['n_warnings'], errors='coerce') > 0).sum()) if n_part else 0
    n_fail = checks.loc[checks['status'] == 'fail', 'file_id'].nunique() if checks is not None and len(checks) else 0
    items = [
        ('html', 'Summary', params_html({
            'Participants in the table': n_part,
            'Lights-off/on source': ', '.join(f'{k}: {v}' for k, v in src_counts.items()) or '-',
            'Sleep-onset rule': cfg.get('onset_rule', ''),
            'Epoch length (s)': cfg.get('epoch_s', ''),
            'MT / custom stage policy': OTHER_STAGE_POLICY,
            'Event counting window': EVENT_COUNT_WINDOW,
            'Event remap': S.get('remap_note', ''),
            'ODI': f"on, depth >= {cfg.get('odi_thr')} %" if cfg.get('odi') else 'off',
            'T90': f"on, SpO2 < {cfg.get('spo2_thr')} % (floor {cfg.get('spo2_floor')} %)" if cfg.get('t90') else 'off',
            'Participants with warnings / failures': f'{n_warn} / {n_fail}',
        })),
        ('fig', 'Time in bed and sleep period', plot_lights_overview(wide) if n_part else None),
        ('fig', 'Sleep macrostructure', plot_metric_distributions(wide, ('macrostructure',),
                                                                  'Sleep macrostructure') if n_part else None),
        ('fig', 'Event indices', plot_metric_distributions(wide, ('respiratory', 'plm', 'arousal', 'odi'),
                                                           'Event indices') if n_part else None),
        ('fig', 'Oxygen saturation', plot_metric_distributions(wide, ('spo2',), 'Oxygen saturation') if n_part else None),
        ('html', 'Consistency checks', checks_summary_html(checks)),
        ('html', 'Participants', participants_html(wide)),
    ]
    if comparison is not None and len(comparison):
        mism = comparison[comparison['status'] == 'mismatch']
        counts = comparison['status'].value_counts().to_dict()
        html = ('<p>' + ', '.join(f'<b>{k}</b>: {v}' for k, v in counts.items()) + '</p>'
                + (mism.round(3).to_html(index=False, border=0) if len(mism) else '<p>No mismatch.</p>'))
        items += [('fig', 'Comparison with the reference table', plot_comparison(comparison)),
                  ('html', 'Comparison - mismatches', html)]
    items += [('html', 'Metric glossary', glossary_html()),
              ('html', 'References', references_html()),
              ('html', 'Where to look', guidance_html(cfg, S['data_dir'], S['reports_dir'],
                                                      has_subject_info=S.get('subj_info') is not None,
                                                      has_comparison=comparison is not None))]
    save_report_html(path, 'Sleep macrostructure - database report', items)
    close_figs(items)
    return path


def finalize_outputs(cfg, failed, log, comparison=None):
    """Rebuild the database tables, the workbook and the report (all non-fatal)."""
    reports_dir = S['reports_dir']
    tables, xlsx = {}, None
    try:
        tables = rebuild_globals(cfg, log)
        # always rewritten (header only when nothing failed) so a stale list never survives a clean run
        pd.DataFrame(failed, columns=['file_id', 'reason']).to_csv(
            reports_dir / 'sleep_metrics_failed.tsv', sep='	', index=False)
    except Exception as exc:
        log(f'&#9888; could not rebuild the database tables: {exc}')
    try:
        xlsx = write_excel(tables, cfg, failed, reports_dir / 'sleep_metrics_database.xlsx', log)
    except Exception as exc:
        log(f'&#9888; could not write the workbook: {exc}')
    try:
        build_database_report(tables, cfg, reports_dir / 'sleep_metrics_database_report.html', comparison)
    except Exception as exc:
        log(f'&#9888; could not write the database report: {exc}')
    return tables, xlsx


def run(_=None):
    """Section-4 entry point: loop over the selected recordings, then rebuild the database outputs."""
    with out_run:
        clear_output()

        def log(msg):
            display(HTML(msg))

        try:
            if not S.get('parts'):
                lbl_run.value = '<span style="color:#c62828">Run the scan in Section 1 first.</span>'
                return
            cfg, errors = get_params()
            if errors:
                lbl_run.value = ('<span style="color:#c62828">' +
                                 '<br>'.join('&#10007; ' + e for e in errors) + '</span>')
                return
            S['cfg'] = cfg
            todo = selected_participants()
            n_skipped = len(participant_selector.skipped_ids())
            if participant_selector.warning_html():          # a subset with already-processed ids
                log(participant_selector.warning_html())
            if not todo:
                lbl_run.value = ('<span style="color:#ef6c00">Nothing to do, every selected '
                                 'recording is already processed (untick Skip to redo them). The database '
                                 'outputs are rebuilt anyway.</span>')
            progress_part.max = max(len(todo), 1)
            progress_part.value = 0
            if todo:
                lbl_run.value = f'<i>Running {len(todo)} recordings...</i>'

            done, failed = [], []
            for i, p in enumerate(todo, 1):
                progress_part.value = i - 1
                progress_part.description = f'Participants: {i}/{len(todo)}'
                try:
                    summary = process_participant(p, cfg, log)
                    p['done'] = True
                    done.append(p['file_id'])
                    log(f'<b>{p["file_id"]}</b>: <small>{summary}</small>')
                except Exception as exc:
                    failed.append((p['file_id'], f'{type(exc).__name__}: {exc}'))
                    log(f'<b>{p["file_id"]}</b> <span style="color:#c62828">&#10007; failed: {exc}</span>')
                progress_part.value = i

            lbl_phase.value = '<small>rebuilding the database tables...</small>'
            tables, xlsx = finalize_outputs(cfg, failed, log)
            lbl_phase.value = ''
            reports_dir = S['reports_dir']
            wide = tables.get('wide')
            n_part = len(wide) if wide is not None else 0
            log(f'<hr><b>{len(done)} processed</b>, {len(failed)} failed, {n_skipped} skipped &mdash; '
                f'the database table now covers <b>{n_part}</b> recordings.')
            log('<div style="background:#e8f5e9;border-left:4px solid #2e7d32;padding:8px 12px;margin:8px 0">'
                f'<b>Start here:</b> <code>{reports_dir / "global_sleep_metrics.tsv"}</code> (one row per '
                f'recording) and <code>{reports_dir / "sleep_metrics_database_report.html"}</code>, whose '
                'closing <i>Where to look</i> section maps every file this run produced.</div>')
            log(f'Data: <code>{S["data_dir"]}</code><br>Reports: <code>{reports_dir}</code>'
                + (f'<br>Workbook: <code>{xlsx}</code>' if xlsx else ''))
            if n_part:
                display(HTML(participants_html(wide)))
            if todo:
                lbl_run.value = (f'<span style="color:#2e7d32">Done, {len(done)} processed, '
                                 f'{len(failed)} failed.</span>')
            build_participant_list()
        except Exception as exc:
            lbl_run.value = f'<span style="color:#c62828">Run failed: {exc}</span>'
            display(HTML(f'<pre style="color:#c62828">{exc}</pre>'))


btn_run.on_click(run)
display(widgets.VBox([widgets.HBox([btn_run, lbl_run]), progress_part,
                      widgets.VBox([progress_step, _STEP_LEGEND]), lbl_phase, out_run]))

## 5. Compare with a reference table (optional)

The same metrics are printed on the clinical PSG reports. Load a table with **one row per participant**
(`.csv` / `.tsv` / `.xlsx`): pick the id column, check the metric → column mapping (auto-guessed from
the column names, `(none)` = not compared) and the tolerance of each metric (default 1 min / 1 %-point /
1 event/h, exact for counts), then **Compare**. The result is written to
`reports_features_macrostructure/comparison_with_reference.tsv` and added to the database report.

A **systematic offset** on one metric points to a definition difference (lights-off/on source, sleep-onset
rule, MT handling, events counted over TIB vs the whole recording). An **isolated mismatch** points to a
scoring or file problem for that participant.

In [ ]:
# =============================================================================
# Section 5 - comparison with an external reference table
# =============================================================================
fc_ref = FileChooser(os.getcwd())
fc_ref.filter_pattern = ['*.csv', '*.tsv', '*.xlsx']
fc_ref.title = '<b>Reference table</b> (one row per participant; e.g. the values of the clinical reports):'
dd_ref_id = widgets.Dropdown(description='Participant id column:', options=[], value=None,
                             layout=widgets.Layout(width='360px'), style=_LBL)
btn_guess = widgets.Button(description='Re-guess mapping', icon='magic', layout=widgets.Layout(width='160px'))
btn_compare = widgets.Button(description='Compare', button_style='success', icon='check')
lbl_compare = widgets.HTML('<i>Run Section 4 first, then load a reference table.</i>')
map_box = widgets.VBox([], layout=widgets.Layout(max_height='420px', overflow_y='auto',
                                                 border='1px solid #ddd', padding='4px'))
map_widgets = {}          # metric key -> (dropdown, tolerance)
out_compare = widgets.Output()


def _build_mapping_rows(ref_columns):
    """One row per metric: label, reference-column dropdown, tolerance field."""
    map_widgets.clear()
    guess = guess_metric_mapping(ref_columns)
    rows = [widgets.HTML('<small style="color:#555">metric &nbsp;|&nbsp; reference column &nbsp;|&nbsp; '
                         'tolerance (absolute, in the metric unit)</small>')]
    for key in METRIC_KEYS:
        dd = widgets.Dropdown(options=['(none)'] + [str(c) for c in ref_columns], value=guess.get(key, '(none)'),
                              layout=widgets.Layout(width='260px'))
        tol = widgets.FloatText(value=default_tolerance(key), layout=widgets.Layout(width='90px'))
        map_widgets[key] = (dd, tol)
        info = METRIC_INFO[key]
        rows.append(widgets.HBox([
            widgets.HTML(f'<div style="width:300px"><b>{info["label"]}</b> '
                         f'<small style="color:#777">({info["unit"]})</small></div>'), dd, tol],
            layout=widgets.Layout(overflow='hidden')))
    map_box.children = tuple(rows)
    n_guessed = sum(1 for k in METRIC_KEYS if guess.get(k, '(none)') != '(none)')
    return n_guessed


def _on_ref(chooser):
    try:
        df, err = load_subject_info(fc_ref.selected)
        if df is None:
            lbl_compare.value = f'<span style="color:#c62828">Reference table unreadable: {err}</span>'
            return
        S['ref_df'] = df
        dd_ref_id.options = list(df.columns)
        guess = [c for c in df.columns if norm(c) in ('file_id', 'fileid', 'sub_id', 'subid', 'id',
                                                       'participant', 'participant_id', 'subject', 'patient')]
        dd_ref_id.value = guess[0] if guess else list(df.columns)[0]
        n = _build_mapping_rows(df.columns)
        lbl_compare.value = (f'<i>Reference loaded: {len(df)} rows, {len(df.columns)} columns &mdash; '
                             f'{n} metric(s) matched automatically; check the mapping, then Compare.</i>')
    except Exception as exc:
        lbl_compare.value = f'<span style="color:#c62828">Reference error: {exc}</span>'


def _on_guess(_=None):
    if S.get('ref_df') is not None:
        n = _build_mapping_rows(S['ref_df'].columns)
        lbl_compare.value = f'<i>{n} metric(s) matched automatically.</i>'


def run_comparison(_=None):
    """Compare global_sleep_metrics.tsv (from disk) with the reference table, write the result,
    and rebuild the database report with the comparison section."""
    with out_compare:
        clear_output()

        def log(msg):
            display(HTML(msg))

        try:
            if not S.get('reports_dir'):
                lbl_compare.value = '<span style="color:#c62828">Run the scan (Section 1) and the run (Section 4) first.</span>'
                return
            gpath = S['reports_dir'] / 'global_sleep_metrics.tsv'
            if not gpath.exists():
                lbl_compare.value = f'<span style="color:#c62828">{gpath} not found - run Section 4 first.</span>'
                return
            ref = S.get('ref_df')
            if ref is None or not dd_ref_id.value:
                lbl_compare.value = '<span style="color:#c62828">Load a reference table first.</span>'
                return
            wide = pd.read_csv(gpath, sep='\t', dtype={'file_id': str})
            mapping = {k: dd.value for k, (dd, _) in map_widgets.items()}
            tol = {k: float(t.value) for k, (_, t) in map_widgets.items()}
            if all(v == '(none)' for v in mapping.values()):
                lbl_compare.value = '<span style="color:#c62828">No metric mapped to a reference column.</span>'
                return
            cmp_df = compare_with_reference(wide, ref, dd_ref_id.value, mapping, tol)
            _write_tsv(cmp_df, S['reports_dir'] / 'comparison_with_reference.tsv')
            S['comparison'] = cmp_df
            counts = cmp_df['status'].value_counts().to_dict()
            log('<b>Comparison:</b> ' + ', '.join(f'{k} {v}' for k, v in counts.items()))
            n_ref_only = len(set(ref[dd_ref_id.value].astype(str).str.strip().map(norm))
                             - set(wide['file_id'].astype(str).map(norm)))
            if n_ref_only:
                log(f'<span style="color:#ef6c00">&#9888; {n_ref_only} participant(s) of the reference '
                    f'table have no row in global_sleep_metrics.tsv (id mismatch or not processed).</span>')
            mism = cmp_df[cmp_df['status'] == 'mismatch']
            if len(mism):
                per_metric = mism.groupby('metric').agg(n=('file_id', 'count'), mean_diff=('diff', 'mean'))
                log('<b>Mismatches per metric</b> (a mean diff far from 0 = systematic definition difference):')
                display(HTML(per_metric.round(2).reset_index().to_html(index=False, border=0)))
                display(HTML(mism.round(3).to_html(index=False, border=0)))
            else:
                log('<span style="color:#2e7d32">No mismatch beyond the tolerances.</span>')
            cfg = S.get('cfg') or get_params()[0]
            try:
                tables = rebuild_globals(cfg, log)
                build_database_report(tables, cfg, S['reports_dir'] / 'sleep_metrics_database_report.html',
                                      comparison=cmp_df)
                log(f'Report updated: <code>{S["reports_dir"] / "sleep_metrics_database_report.html"}</code>'
                    f'<br>Table: <code>{S["reports_dir"] / "comparison_with_reference.tsv"}</code>')
            except Exception as exc:
                log(f'&#9888; comparison written but the report could not be rebuilt: {exc}')
            lbl_compare.value = (f'<span style="color:#2e7d32">Done, {counts.get("match", 0)} match, '
                                 f'{counts.get("mismatch", 0)} mismatch.</span>')
        except Exception as exc:
            lbl_compare.value = f'<span style="color:#c62828">Comparison failed: {exc}</span>'
            display(HTML(f'<pre style="color:#c62828">{exc}</pre>'))


fc_ref.register_callback(_on_ref)
btn_guess.on_click(_on_guess)
btn_compare.on_click(run_comparison)
display(widgets.VBox([fc_ref, widgets.HBox([dd_ref_id, btn_guess]), map_box,
                      widgets.HBox([btn_compare, lbl_compare]), out_compare]))

## Outputs: where to look

**Start with** `reports_features_macrostructure/global_sleep_metrics.tsv` (one row per recording, the
table for statistics) and the database report `sleep_metrics_database_report.html`, whose closing
*Where to look* section repeats this guide.

| File | What it holds | How to read it |
|---|---|---|
| `reports_features_macrostructure/global_sleep_metrics.tsv` | `file_id`, one column per metric (`tib_min`, `sol_min`, `spt_min`, `tst_min`, `waso_min`, `se_pct`, `sme_pct`, `lat_*_min`, `n1_pct` … `rem_pct`, `ahi` / `oai` / `cai` / `mai` / `hi`, `plm_index`, `arousal_index`, `odi`, `t90_*`, `t88_sleep_min`, `spo2_*`, plus the raw event counts `n_*`), then the provenance columns (`lights_source`, `lights_off_clock`, `lights_on_clock`, `sleep_onset_rule`, `n_mt_custom`, `event_source`, `n_unmapped_events`, `odi_rule`, `n_warnings` …) and, when a participant table was given, its columns joined on the right. Rebuilt from every per-recording file on disk at each run. | Units: minutes, %, events/h, counts (see the glossary). **Before trusting a value**: `lights_source = recording_bounds` means TIB is the whole recording (SOL and SE biased), `n_unmapped_events > 0` means events absent from `event_remap.json` were *not* counted in the indices (map them with tool 4, re-run with *Skip* unticked), `n_warnings > 0` → read the checks file. Group by any participant-info column for population comparisons. |
| `reports_features_macrostructure/global_sleep_metrics_checks.tsv` | Every consistency check of every recording: `check`, `status` (`ok` / `info` / `warning` / `fail`), `detail`. | A `fail` means a metric could not be computed or trusted (`no_sleep`, `hypno_vs_edf_length`), a `warning` flags a fallback or an unusual situation (`lights_fallback`, `no_lights_on`, `mt_custom_present`, `unmapped_events`, `spo2_artifact`, `odi_rule` …). |
| `reports_features_macrostructure/sleep_metrics_database.xlsx` | Sheets `sleep_metrics`, `checks`, `glossary` (definitions + indicative ranges), `parameters` (this run), `failed`. | Convenience copy, the TSV files are the source of truth. |
| `reports_features_macrostructure/sleep_metrics_database_report.html` | Time-in-bed overview per recording (coloured by lights source), distribution of every metric with the indicative normal range shaded, checks summary, participant table, comparison (if run), glossary, *Where to look*. | The entry point for a first look at a database. |
| `reports_features_macrostructure/sleep_metrics_failed.tsv` | Recordings that produced no metrics and why. | Missing hypnogram, hypnogram/EDF length mismatch > 1 epoch, unreadable EDF header. |
| `reports_features_macrostructure/comparison_with_reference.tsv` | One row per (recording, compared metric): `ours`, `reference`, `ref_column`, `diff`, `tol`, `status` (`match` / `mismatch` / `missing_ours` / `missing_reference` / `not_in_reference_table`). | Filter on `mismatch`, a mean `diff` far from 0 on one metric = definition difference, an isolated outlier = scoring or file problem. |
| `derivatives/features_macrostructure/<subtree>/{file_id}_sleep_metrics.tsv` | Long format: `group`, `metric`, `value`, `unit`, `definition_short` (metrics + provenance rows). The skip marker: delete it to reprocess that recording. | `df.pivot(index='file_id', columns='metric', values='value')` gives the wide row. |
| `derivatives/features_macrostructure/<subtree>/{file_id}_sleep_metrics_checks.tsv` | The recording's checks (same columns as the global checks file). | Written before the metrics file, so an interrupted run leaves no metrics without its checks. |

In [ ]:
# Collapsible section: where the indicative values come from (same content as the database report)
display(HTML('<details><summary style="cursor:pointer; font-size:1.25em; font-weight:bold">'
             'Where the indicative values come from - references (click to open)</summary>'
             + references_html() + '</details>'))